# 01 — Nettoyage des données

On part des fichiers bruts Binance (`data/raw/BTCUSDT-1h-*.zip`) et on produit un dataset propre (`data/processed/btc_clean.csv`) prêt pour les features.

Objectif : bons types, colonnes utiles, **trié par date croissante**, sans doublons ni trous horaires.

## Partie 1 — Chargement des fichiers bruts

Les données Binance arrivent en 6 fichiers `.zip` mensuels, **sans en-tête**. On les lit un par un, on leur donne les bons noms de colonnes, puis on les empile en un seul dataset.

In [26]:
import pandas as pd


In [27]:
from pathlib import Path

RAW_DIR = Path("../data/raw")

files = sorted(RAW_DIR.glob("BTCUSDT-1h-*.csv"))

print(f"Nombre de fichiers trouvés : {len(files)}")

for f in files:
    print(f.name)

Nombre de fichiers trouvés : 12
BTCUSDT-1h-2025-07.csv
BTCUSDT-1h-2025-08.csv
BTCUSDT-1h-2025-09.csv
BTCUSDT-1h-2025-10.csv
BTCUSDT-1h-2025-11.csv
BTCUSDT-1h-2025-12.csv
BTCUSDT-1h-2026-01.csv
BTCUSDT-1h-2026-02.csv
BTCUSDT-1h-2026-03.csv
BTCUSDT-1h-2026-04.csv
BTCUSDT-1h-2026-05.csv
BTCUSDT-1h-2026-06.csv


In [28]:
columns = [
    "open_time",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "close_time",
    "quote_asset_volume",
    "number_of_trades",
    "taker_buy_base_volume",
    "taker_buy_quote_volume",
    "ignore"
]

In [29]:
dfs = []

In [30]:
for file in files:
    temp_df = pd.read_csv(file, header=None)
    temp_df.columns = columns
    dfs.append(temp_df)
df = pd.concat(dfs, ignore_index=True)
print("Shape du dataset :", df.shape)
df.head()

Shape du dataset : (8760, 12)


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume,ignore
0,1751328000000000,107146.51,107449.25,107026.93,107377.02,253.62802,1751331599999999,2.720125e+07,68242,114.96074,1.232970e+07,0
1,1751331600000000,107377.03,107540.00,107171.42,107220.00,219.64653,1751335199999999,2.358859e+07,67588,80.21028,8.614095e+06,0
2,1751335200000000,107220.00,107410.74,107097.87,107101.69,207.06087,1751338799999999,2.220821e+07,54684,81.41164,8.731958e+06,0
3,1751338800000000,107101.68,107223.00,107070.20,107192.38,293.12458,1751342399999999,3.141034e+07,38802,152.24013,1.631451e+07,0
4,1751342400000000,107192.39,107203.11,106867.43,106881.60,261.75371,1751345999999999,2.801826e+07,42983,89.94461,9.629883e+06,0


## Partie 2 — Conversion des types

Les colonnes de temps sont en microsecondes (entiers) → on les passe en `datetime`. On force aussi les prix/volumes en `float` et `number_of_trades` en `int`.

In [31]:
df["open_time"] = pd.to_datetime(df["open_time"], unit="us")
df["close_time"] = pd.to_datetime(df["close_time"], unit="us")

df[["open_time", "close_time"]].head()

,open_time,close_time
0,2025-07-01 00:00:00,2025-07-01 00:59:59.999999
1,2025-07-01 01:00:00,2025-07-01 01:59:59.999999
2,2025-07-01 02:00:00,2025-07-01 02:59:59.999999
3,2025-07-01 03:00:00,2025-07-01 03:59:59.999999
4,2025-07-01 04:00:00,2025-07-01 04:59:59.999999


In [32]:
df.dtypes

open_time                 datetime64[us]
open                             float64
high                             float64
low                              float64
close                            float64
volume                           float64
close_time                datetime64[us]
quote_asset_volume               float64
number_of_trades                   int64
taker_buy_base_volume            float64
taker_buy_quote_volume           float64
ignore                             int64
dtype: object

In [33]:
float_cols = [
    "open",
    "high",
    "low",
    "close",
    "volume",
    "quote_asset_volume",
    "taker_buy_base_volume",
    "taker_buy_quote_volume"
]

df[float_cols] = df[float_cols].astype(float)

In [34]:
df["number_of_trades"] = df["number_of_trades"].astype(int)

## Partie 3 — Contrôles & nettoyage

On vérifie que les heures se suivent (écart régulier de 1h), on supprime la colonne `ignore` (inutile), on **trie par date croissante** (critique) et on retire les doublons d'horodatage.

In [35]:
time_diff = df["open_time"].diff()

time_diff.value_counts()

open_time
0 days 01:00:00    8759
Name: count, dtype: int64

In [36]:
df = df.drop(columns=["ignore"])

In [37]:
df.columns

Index(['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time',
       'quote_asset_volume', 'number_of_trades', 'taker_buy_base_volume',
       'taker_buy_quote_volume'],
      dtype='str')

In [38]:
df = df.sort_values("open_time").reset_index(drop=True)

In [39]:
df = df.drop_duplicates(subset=["open_time"]).reset_index(drop=True)

## Partie 4 — Vérifications finales

Dernier contrôle avant sauvegarde : aucun `NaN`, dates bien triées, bons types, et aucun trou horaire.

In [40]:
df.isna().sum()

open_time                 0
open                      0
high                      0
low                       0
close                     0
volume                    0
close_time                0
quote_asset_volume        0
number_of_trades          0
taker_buy_base_volume     0
taker_buy_quote_volume    0
dtype: int64

In [41]:
df["open_time"].is_monotonic_increasing

True

In [42]:
df.dtypes

open_time                 datetime64[us]
open                             float64
high                             float64
low                              float64
close                            float64
volume                           float64
close_time                datetime64[us]
quote_asset_volume               float64
number_of_trades                   int64
taker_buy_base_volume            float64
taker_buy_quote_volume           float64
dtype: object

In [43]:
missing_hours = df[time_diff > pd.Timedelta(hours=1)]

missing_hours[["open_time"]]

,open_time


## Partie 5 — Sauvegarde

Le dataset propre est enregistré dans `data/processed/btc_clean.csv`. C'est le point de départ des features (Partie 2 du projet, faite par Lina).

In [44]:
df.to_csv("../data/processed/btc_clean.csv", index=False)

print("Dataset sauvegardé")

Dataset sauvegardé


In [45]:
print("Shape:", df.shape)
print("\nColonnes:")
print(df.columns.tolist())

print("\nTypes:")
print(df.dtypes)

print("\nValeurs manquantes:")
print(df.isna().sum())

print("\nDoublons open_time:", df.duplicated(subset=["open_time"]).sum())

print("\nDates triées:", df["open_time"].is_monotonic_increasing)

print("\nPremières lignes:")
display(df.head())

print("\nDernières lignes:")
display(df.tail())

Shape: (8760, 11)

Colonnes:
['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_asset_volume', 'number_of_trades', 'taker_buy_base_volume', 'taker_buy_quote_volume']

Types:
open_time                 datetime64[us]
open                             float64
high                             float64
low                              float64
close                            float64
volume                           float64
close_time                datetime64[us]
quote_asset_volume               float64
number_of_trades                   int64
taker_buy_base_volume            float64
taker_buy_quote_volume           float64
dtype: object

Valeurs manquantes:
open_time                 0
open                      0
high                      0
low                       0
close                     0
volume                    0
close_time                0
quote_asset_volume        0
number_of_trades          0
taker_buy_base_volume     0
taker_buy_quote_volume    0
dtype:

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume
0,2025-07-01 00:00:00,107146.51,107449.25,107026.93,107377.02,253.62802,2025-07-01 00:59:59.999999,2.720125e+07,68242,114.96074,1.232970e+07
1,2025-07-01 01:00:00,107377.03,107540.00,107171.42,107220.00,219.64653,2025-07-01 01:59:59.999999,2.358859e+07,67588,80.21028,8.614095e+06
2,2025-07-01 02:00:00,107220.00,107410.74,107097.87,107101.69,207.06087,2025-07-01 02:59:59.999999,2.220821e+07,54684,81.41164,8.731958e+06
3,2025-07-01 03:00:00,107101.68,107223.00,107070.20,107192.38,293.12458,2025-07-01 03:59:59.999999,3.141034e+07,38802,152.24013,1.631451e+07
4,2025-07-01 04:00:00,107192.39,107203.11,106867.43,106881.60,261.75371,2025-07-01 04:59:59.999999,2.801826e+07,42983,89.94461,9.629883e+06



Dernières lignes:


,open_time,open,high,low,close,volume,close_time,quote_asset_volume,number_of_trades,taker_buy_base_volume,taker_buy_quote_volume
8755,2026-06-30 19:00:00,58680.86,58839.16,58586.01,58818.05,678.64150,2026-06-30 19:59:59.999999,3.983733e+07,160151,329.59967,1.934946e+07
8756,2026-06-30 20:00:00,58818.00,58874.99,58648.26,58732.01,428.82351,2026-06-30 20:59:59.999999,2.519593e+07,107496,190.70232,1.120409e+07
8757,2026-06-30 21:00:00,58732.00,58770.00,58509.99,58639.99,398.14222,2026-06-30 21:59:59.999999,2.333860e+07,106726,190.26444,1.115222e+07
8758,2026-06-30 22:00:00,58639.99,58806.91,58513.80,58607.99,274.43078,2026-06-30 22:59:59.999999,1.609680e+07,82205,135.70062,7.960798e+06
8759,2026-06-30 23:00:00,58607.99,58664.00,58555.00,58624.71,551.65078,2026-06-30 23:59:59.999999,3.233995e+07,76224,234.24755,1.373100e+07


In [46]:
Path("../data/processed/btc_clean.csv").exists()

True